# EXP_v4: MFI + MLE (Python)

`Test_MFI_EAP_unif_on_the_simulated_bank.ipynb` の MLE 版です。DQN と MFI を極力同条件で比較するため、`Train_and_test_DQN_on_the_simulated_banks.ipynb` と同じ `RESPOND`、`FI`、`MLE`、`MLE_TEST` を使用します。

被験者を step ごとに一括処理する乱数生成順も DQN の `TEST` に揃えています。各 step では、現在の MLE 推定値における Fisher 情報量が最大の未出題項目を選択します。全問正解・全問不正解中は DQN と同じく、現在値から項目困難度の最大値・最小値へ半分移動します。出力ファイル名には既定で `_python` を付けます。

In [37]:
# -*- coding: utf-8 -*-
from dataclasses import dataclass
from pathlib import Path
from typing import Any, cast

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar


def find_project_root():
    candidates = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Colab Notebooks/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "EXP_v4" / "data" / "uncorrelated_banks").is_dir():
            return root

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "EXP_v4" / "data" / "uncorrelated_banks").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. "
        "In Colab, place the repository at MyDrive/Grad_Research or /content/Grad_Research."
    )


ROOT = find_project_root()
RESULTS_DIR = ROOT / "EXP_v4" / "results"

print(f"Project root: {ROOT}")
print(f"Results dir : {RESULTS_DIR}")

Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v4/results


In [38]:
@dataclass
class Config:
    test_length: int = 10

    # Bank / evaluation data
    bank_type: str = "uncor"  # 'uncor' | 'cor'
    bank_id: int = 1
    n_items: int = 400
    testing_size: int = 0  # 0: use all theta values
    theta_csv: str = ""  # empty: EXP_v4/data/theta_true/theta_true_{bank_id}.csv

    # Reproducibility / output
    seed: int = 20260430
    output_suffix: str = "_python"

In [39]:
# These functions are aligned with the EXP_v4 DQN notebook.
def RESPOND(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    p = (1 - c) / (1 + np.exp(-D * a * (theta - b))) + c
    return (np.random.random(size=p.shape) <= p).astype(int)


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    return (
        D**2
        * a**2
        * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )


def MLE(item_paras, resp, D=1):
    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]

    def mins_likelihood(x):
        logl = 0
        for i in range(len(resp)):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp[i] * np.log(p) + (1 - resp[i]) * np.log(1 - p)
        return logl

    result = cast(
        Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded")
    )
    return np.array(result.x).reshape(
        1,
    )


def MLE_TEST(item_paras, resp, D=1):
    def mins_likelihood(x):
        logl = 0
        for i in range(resp_i.shape[0]):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp_i[i] * np.log(p) + (1 - resp_i[i]) * np.log(1 - p)
        return logl

    theta = np.zeros(resp.shape[1])
    for i in range(resp.shape[1]):
        resp_i = resp[:, i]
        a = item_paras[:, i, 0]
        b = item_paras[:, i, 1]
        c = item_paras[:, i, 2]
        result = cast(
            Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded")
        )
        theta[i] = result.x
    return np.expand_dims(theta, axis=0)

In [40]:
def choose_mfi(item_bank, theta_current, item_ids):
    """Choose each subject's unadministered item with maximum FI."""
    information = np.vstack([FI(item_bank, theta) for theta in theta_current])

    if item_ids.shape[0] > 0:
        subject_indices = np.arange(len(theta_current))[:, None]
        information[subject_indices, item_ids.T] = -np.inf

    return information.argmax(axis=1).astype(np.int64)


def estimate_theta_mle(item_bank, item_ids, responses, current_theta):
    testing_size = responses.shape[1]
    theta_hat = np.zeros(testing_size)
    idx_full = np.sum(responses, axis=0) == responses.shape[0]
    idx_zero = np.sum(responses, axis=0) == 0
    idx_norm = ~(idx_full | idx_zero)

    theta_hat[idx_full] = (
        current_theta[idx_full] + (item_bank[:, 1].max() - current_theta[idx_full]) / 2
    )
    theta_hat[idx_zero] = (
        current_theta[idx_zero] - (current_theta[idx_zero] - item_bank[:, 1].min()) / 2
    )
    if np.any(idx_norm):
        theta_hat[idx_norm] = np.squeeze(
            MLE_TEST(
                item_bank[item_ids[:, idx_norm]],
                responses[:, idx_norm],
            )
        )
    return theta_hat


def summarize_steps(theta_true, theta_history):
    rows = []
    theta_true_sd = np.std(theta_true, ddof=1)

    for step, theta_est in enumerate(theta_history, start=1):
        bias = theta_est - theta_true
        theta_est_sd = np.std(theta_est, ddof=1)
        correlation = (
            np.nan
            if theta_true_sd == 0 or theta_est_sd == 0
            else np.corrcoef(theta_true, theta_est)[0, 1]
        )
        rows.append(
            {
                "step": step,
                "Bias": np.mean(bias),
                "RMSE": np.sqrt(np.mean(bias**2)),
                "MAE": np.mean(np.abs(bias)),
                "r": correlation,
            }
        )

    return pd.DataFrame(rows)


def run_mfi(cfg, item_bank, theta_true):
    # Use the same NumPy RNG and subject-vectorized order as DQN TEST.
    np.random.seed(cfg.seed)
    testing_size = len(theta_true)

    theta_current = np.random.rand(testing_size) - 0.5
    item_ids = np.empty((0, testing_size), dtype=np.int64)
    responses = np.empty((0, testing_size), dtype=np.int64)
    theta_history = np.empty((0, testing_size), dtype=float)

    for step in range(cfg.test_length):
        selected = choose_mfi(item_bank, theta_current, item_ids)
        step_responses = RESPOND(item_bank[selected], theta_true)

        item_ids = np.concatenate((item_ids, selected[np.newaxis, :]))
        responses = np.concatenate((responses, step_responses[np.newaxis, :]))
        theta_current = estimate_theta_mle(
            item_bank, item_ids, responses, theta_current
        )

        theta_history = np.concatenate((theta_history, theta_current[np.newaxis, :]))
        bias = theta_current - theta_true
        print(
            "step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}".format(
                step + 1,
                np.mean(bias),
                np.sqrt(np.mean(bias**2)),
                np.mean(np.abs(bias)),
            )
        )

    user_id_col = np.repeat(np.arange(1, testing_size + 1), cfg.test_length)
    step_col = np.tile(np.arange(1, cfg.test_length + 1), testing_size)
    records = pd.DataFrame(
        {
            "userID": user_id_col,
            "step": step_col,
            "itemID": (item_ids + 1).T.reshape(-1),
            "resp": responses.T.reshape(-1),
            "theta_true": np.repeat(theta_true, cfg.test_length),
            "theta_est": theta_history.T.reshape(-1),
            "bias": (theta_history - theta_true).T.reshape(-1),
        }
    )
    summary_by_step = summarize_steps(theta_true, theta_history)
    return records, summary_by_step

In [41]:
cfg = Config(
    test_length=10,
    bank_type="uncor",
    bank_id=1,
    n_items=200,
    testing_size=0,
    theta_csv="",
    seed=20260430,
    output_suffix="_python",
)

bank_dir = {
    "uncor": ROOT / "EXP_v4" / "data" / "uncorrelated_banks",
    "cor": ROOT / "EXP_v4" / "data" / "correlated_banks",
}.get(cfg.bank_type)
if bank_dir is None:
    raise ValueError("bank_type must be 'uncor' or 'cor'.")

bank_path = bank_dir / f"item_bank_{cfg.bank_type}_{cfg.bank_id}.csv"
item_bank = pd.read_csv(bank_path)[["a", "b", "c"]].to_numpy()[: cfg.n_items]

if cfg.theta_csv:
    theta_path = Path(cfg.theta_csv).expanduser()
    if not theta_path.is_absolute():
        theta_path = ROOT / theta_path
else:
    theta_path = (
        ROOT / "EXP_v4" / "data" / "theta_true" / f"theta_true_{cfg.bank_id}.csv"
    )

theta_true = pd.read_csv(theta_path)["x"].to_numpy()
if not cfg.theta_csv and cfg.testing_size > 0:
    theta_true = theta_true[: cfg.testing_size]

if cfg.test_length > len(item_bank):
    raise ValueError("test_length cannot exceed the number of items in the bank.")
if len(theta_true) < 2:
    raise ValueError("At least two theta values are required to calculate correlation.")

print(f"item bank  : {item_bank.shape} ({bank_path})")
print(f"theta_true : {theta_true.shape} ({theta_path})")
print(f"Config     : {cfg}")

item bank  : (200, 3) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v4/data/uncorrelated_banks/item_bank_uncor_1.csv)
theta_true : (5000,) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v4/data/theta_true/theta_true_1.csv)
Config     : Config(test_length=10, bank_type='uncor', bank_id=1, n_items=200, testing_size=0, theta_csv='', seed=20260430, output_suffix='_python')


In [42]:
records, summary_by_step = run_mfi(cfg, item_bank, theta_true)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
stem = (
    f"{cfg.bank_type}_{cfg.bank_id}_{cfg.n_items}items_"
    f"MFI_MLE{cfg.output_suffix}"
)
records_path = RESULTS_DIR / f"records_{stem}.csv"
summary_path = RESULTS_DIR / f"summary_{stem}.csv"
records.to_csv(records_path, index=False)
summary_by_step.to_csv(summary_path, index=False)

display(summary_by_step.tail(1))
print(f"Saved records to: {records_path}")
print(f"Saved summary to: {summary_path}")

step 1, bias 0.355, rmse 1.217, mae 0.993
step 2, bias 0.266, rmse 1.129, mae 0.897
step 3, bias 0.169, rmse 1.024, mae 0.802
step 4, bias 0.106, rmse 1.040, mae 0.786
step 5, bias 0.126, rmse 0.914, mae 0.694
step 6, bias 0.101, rmse 0.852, mae 0.645
step 7, bias 0.082, rmse 0.820, mae 0.609
step 8, bias 0.075, rmse 0.754, mae 0.568
step 9, bias 0.061, rmse 0.717, mae 0.535
step 10, bias 0.058, rmse 0.665, mae 0.503


,step,Bias,RMSE,MAE,r
9,10,0.057981,0.665106,0.503059,0.834876


Saved records to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v4/results/records_uncor_1_200items_MFI_MLE_python.csv
Saved summary to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v4/results/summary_uncor_1_200items_MFI_MLE_python.csv
